# Análise Exploratória de Dados (EDA) - Dataset de Churn

## Objetivo
Análise abrangente do dataset de churn de clientes de telecom para identificar padrões, distribuições e relações entre variáveis.

## 1. Setup e Importações

In [ ]:
# Importações essenciais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Para análises estatísticas
from scipy import stats
from scipy.stats import skew, kurtosis
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Configuração de plotagem
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

## 2. Carregamento e Inspeção dos Dados

In [ ]:
# Carregamento do dataset de treino
df = pd.read_csv('./datasets/churn_train.csv')

print(f"Dimensões do dataset: {df.shape[0]} linhas × {df.shape[1]} colunas")

print(f"Tamanho do arquivo: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

print("\nprimeiras linhas:")
display(df.head())

print("\núltimas linhas:")
display(df.tail())


In [ ]:

print(df.info())
display(df.describe())

## 3. Análise de Valores Nulos

In [ ]:
# Análise de valores nulos
print("=" * 80)
print("ANÁLISE DE VALORES NULOS")
print("=" * 80)

missing_data = pd.DataFrame({
    'Coluna': df.columns,
    'Valores Nulos': df.isnull().sum(),
    'Percentual (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'Tipo': df.dtypes
})

missing_data = missing_data.sort_values('Percentual (%)', ascending=False)
display(missing_data)

print(f"\n✅ Total de valores nulos no dataset: {df.isnull().sum().sum()}")

# Visualização de valores nulos
fig, ax = plt.subplots(figsize=(14, 6))
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]  # Mostrar apenas as colunas com valores nulos

if len(missing_pct) > 0:
    missing_pct.plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('Percentual de Valores Nulos (%)')
    ax.set_title('Distribuição de Valores Nulos por Coluna', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("\n✅ Nenhum valor nulo encontrado no dataset!")

## 4. Classificação de Variáveis

In [ ]:
print("=" * 80)
print("CLASSIFICAÇÃO DE VARIÁVEIS")
print("=" * 80)

# Separar variáveis por tipo
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"\n📊 VARIÁVEIS NUMÉRICAS ({len(numerical_cols)}):")
print(numerical_cols)

print(f"\n📝 VARIÁVEIS CATEGÓRICAS ({len(categorical_cols)}):")
print(categorical_cols)

# Classificação adicional: Discretas vs Contínuas
discrete_numerical = []
continuous_numerical = []

for col in numerical_cols:
    unique_values = df[col].nunique()
    if unique_values <= 20:  # Heurística para discrete
        discrete_numerical.append(col)
    else:
        continuous_numerical.append(col)

print(f"\n🔢 VARIÁVEIS NUMÉRICAS DISCRETAS ({len(discrete_numerical)}):")
print(discrete_numerical)

print(f"\n📈 VARIÁVEIS NUMÉRICAS CONTÍNUAS ({len(continuous_numerical)}):")
print(continuous_numerical)

# Classificação por contexto
demographic_vars = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure']
financial_vars = ['MonthlyCharges', 'TotalCharges']
service_vars = [col for col in categorical_cols if col not in demographic_vars and col != 'Churn']

print(f"\n👥 VARIÁVEIS DEMOGRÁFICAS:")
print([v for v in demographic_vars if v in df.columns])

print(f"\n💰 VARIÁVEIS FINANCEIRAS:")
print([v for v in financial_vars if v in df.columns])

print(f"\n🔧 VARIÁVEIS DE SERVIÇO:")
print(service_vars)

print(f"\n🎯 VARIÁVEL ALVO (Target):")
print(['Churn'])

## 5. Análise Univariada - Variáveis Numéricas

In [ ]:
# Estatísticas descritivas para variáveis numéricas
print("=" * 80)
print("ESTATÍSTICAS DESCRITIVAS - VARIÁVEIS NUMÉRICAS")
print("=" * 80)

stats_df = pd.DataFrame({
    'Variável': numerical_cols,
    'Média': [df[col].mean() for col in numerical_cols],
    'Mediana': [df[col].median() for col in numerical_cols],
    'Desvio Padrão': [df[col].std() for col in numerical_cols],
    'Mínimo': [df[col].min() for col in numerical_cols],
    'Máximo': [df[col].max() for col in numerical_cols],
    'Skewness': [skew(df[col].dropna()) for col in numerical_cols],
    'Kurtosis': [kurtosis(df[col].dropna()) for col in numerical_cols]
}).round(3)

display(stats_df)

# Visualização: Distribuições de variáveis numéricas
fig, axes = plt.subplots(len(numerical_cols), 2, figsize=(14, 4*len(numerical_cols)))
if len(numerical_cols) == 1:
    axes = [axes]

for idx, col in enumerate(numerical_cols):
    # Histograma com KDE
    axes[idx, 0].hist(df[col], bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    axes[idx, 0].set_title(f'Distribuição: {col}', fontweight='bold')
    axes[idx, 0].set_xlabel(col)
    axes[idx, 0].set_ylabel('Frequência')
    
    # Box plot
    axes[idx, 1].boxplot(df[col], vert=False)
    axes[idx, 1].set_title(f'Box Plot: {col}', fontweight='bold')
    axes[idx, 1].set_xlabel(col)
    
plt.tight_layout()
plt.show()

# Visualização: Densidade (KDE)
fig, axes = plt.subplots(1, len(numerical_cols), figsize=(14, 4))
if len(numerical_cols) == 1:
    axes = [axes]

for idx, col in enumerate(numerical_cols):
    df[col].plot(kind='density', ax=axes[idx], color='steelblue', linewidth=2)
    axes[idx].set_title(f'Densidade: {col}', fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    
plt.tight_layout()
plt.show()

## 6. Análise Univariada - Variáveis Categóricas

In [ ]:
print("=" * 80)
print("ANÁLISE DE VARIÁVEIS CATEGÓRICAS")
print("=" * 80)

# Análise individual de cada variável categórica
for col in categorical_cols:
    print(f"\n📊 {col.upper()}:")
    print(f"  • Valores únicos: {df[col].nunique()}")
    print(f"  • Modo (Moda): {df[col].mode()[0] if len(df[col].mode()) > 0 else 'N/A'}")
    print(f"  • Distribuição:")
    print(df[col].value_counts().to_string())
    print()

# Visualização de variáveis categóricas
fig, axes = plt.subplots((len(categorical_cols) + 1) // 2, 2, figsize=(14, 4 * ((len(categorical_cols) + 1) // 2)))
axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

for idx, col in enumerate(categorical_cols):
    value_counts = df[col].value_counts()
    axes[idx].bar(range(len(value_counts)), value_counts.values, color='steelblue', alpha=0.7, edgecolor='black')
    axes[idx].set_xticks(range(len(value_counts)))
    axes[idx].set_xticklabels(value_counts.index, rotation=45, ha='right')
    axes[idx].set_title(f'Frequência: {col}', fontweight='bold')
    axes[idx].set_ylabel('Contagem')
    
    # Adicionar valores nas barras
    for i, v in enumerate(value_counts.values):
        axes[idx].text(i, v + 10, str(v), ha='center', va='bottom', fontweight='bold')

# Remover eixos vazios
for idx in range(len(categorical_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

## 7. Detecção de Outliers

In [ ]:
print("=" * 80)
print("DETECÇÃO DE OUTLIERS (Método IQR)")
print("=" * 80)

# Método IQR para detecção de outliers
outlier_summary = []

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_pct = (len(outliers) / len(df)) * 100
    
    outlier_summary.append({
        'Variável': col,
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'Limite Inferior': lower_bound,
        'Limite Superior': upper_bound,
        'Qtd. Outliers': len(outliers),
        'Percentual (%)': outlier_pct
    })
    
    print(f"\n📊 {col}:")
    print(f"  Q1: {Q1:.2f}, Q3: {Q3:.2f}, IQR: {IQR:.2f}")
    print(f"  Limites: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Outliers: {len(outliers)} ({outlier_pct:.2f}%)")

outlier_df = pd.DataFrame(outlier_summary).round(3)
print("\n📋 RESUMO DE OUTLIERS:")
display(outlier_df)

# Visualização de outliers
fig, axes = plt.subplots(1, len(numerical_cols), figsize=(14, 5))
if len(numerical_cols) == 1:
    axes = [axes]

for idx, col in enumerate(numerical_cols):
    box_plot = axes[idx].boxplot(df[col], vert=True, patch_artist=True, widths=0.5)
    
    # Colorir as boxes
    box_plot['boxes'][0].set_facecolor('lightblue')
    
    axes[idx].scatter([1] * len(df[col]), df[col], alpha=0.5, s=30, color='steelblue')
    axes[idx].set_title(f'Outliers: {col}', fontweight='bold')
    axes[idx].set_ylabel('Valor')
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 8. Análise de Separabilidade de Classes (Churn)

In [ ]:
print("=" * 80)
print("DISTRIBUIÇÃO DA VARIÁVEL ALVO (Churn)")
print("=" * 80)

churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print(f"\n📊 Distribuição absoluta:")
print(churn_counts)
print(f"\n📊 Distribuição percentual:")
print(churn_pct)

# Visualização da classe
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Gráfico de barras
axes[0].bar(churn_counts.index, churn_counts.values, color=['green', 'red'], alpha=0.7, edgecolor='black')
axes[0].set_title('Distribuição de Churn (Absolutas)', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Contagem')
axes[0].set_xlabel('Churn')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Gráfico de pizza
colors = ['green', 'red']
axes[1].pie(churn_counts.values, labels=churn_counts.index, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Proporção de Churn', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

# Análise de separabilidade por variáveis numéricas
print("\n" + "=" * 80)
print("SEPARABILIDADE POR VARIÁVEIS NUMÉRICAS")
print("=" * 80)

sep_numeric = []
for col in numerical_cols:
    churned = df[df['Churn'] == 'Yes'][col]
    not_churned = df[df['Churn'] == 'No'][col]
    
    # Teste estatístico (Mann-Whitney U)
    statistic, p_value = stats.mannwhitneyu(churned, not_churned)
    
    sep_numeric.append({
        'Variável': col,
        'Média (Churned)': churned.mean(),
        'Média (Not Churned)': not_churned.mean(),
        'Std (Churned)': churned.std(),
        'Std (Not Churned)': not_churned.std(),
        'P-Value': p_value,
        'Significativo': 'Sim' if p_value < 0.05 else 'Não'
    })

sep_numeric_df = pd.DataFrame(sep_numeric).round(4)
display(sep_numeric_df)

# Visualização: Box plots comparativos
fig, axes = plt.subplots(1, len(numerical_cols), figsize=(14, 5))
if len(numerical_cols) == 1:
    axes = [axes]

for idx, col in enumerate(numerical_cols):
    df.boxplot(column=col, by='Churn', ax=axes[idx])
    axes[idx].set_title(f'{col} vs Churn', fontweight='bold')
    axes[idx].set_xlabel('Churn')
    axes[idx].set_ylabel(col)
    plt.sca(axes[idx])
    plt.xticks(rotation=0)

plt.suptitle('Separabilidade de Classes - Variáveis Numéricas', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Análise de separabilidade por variáveis categóricas
print("\n" + "=" * 80)
print("SEPARABILIDADE POR VARIÁVEIS CATEGÓRICAS")
print("=" * 80)

sep_categorical = []
for col in categorical_cols:
    contingency = pd.crosstab(df[col], df['Churn'])
    chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
    
    sep_categorical.append({
        'Variável': col,
        'Chi2': chi2,
        'P-Value': p_value,
        'Significativo': 'Sim' if p_value < 0.05 else 'Não'
    })

sep_categorical_df = pd.DataFrame(sep_categorical).round(4)
display(sep_categorical_df)

# Visualização: Stacked bar plots
fig, axes = plt.subplots((len(categorical_cols) + 2) // 3, 3, figsize=(16, 4 * ((len(categorical_cols) + 2) // 3)))
axes = axes.flatten() if isinstance(axes, np.ndarray) else [axes]

for idx, col in enumerate(categorical_cols):
    crosstab = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    crosstab.plot(kind='bar', stacked=False, ax=axes[idx], color=['green', 'red'], alpha=0.7)
    axes[idx].set_title(f'{col} vs Churn (%)', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Percentual (%)')
    axes[idx].legend(title='Churn')
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45, ha='right')

for idx in range(len(categorical_cols), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Separabilidade de Classes - Variáveis Categóricas', fontweight='bold', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

## 9. Análise de Relações - Demográficas vs Financeiras

In [ ]:
print("=" * 80)
print("RELAÇÕES DEMOGRÁFICAS vs FINANCEIRAS")
print("=" * 80)

# Relação: Tenure vs Monthly Charges por SeniorCitizen
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, demo_var in enumerate(['SeniorCitizen', 'Partner']):
    for value in df[demo_var].unique():
        subset = df[df[demo_var] == value]
        axes[idx].scatter(subset['tenure'], subset['MonthlyCharges'], 
                         label=f'{demo_var}={value}', alpha=0.6, s=50)
    
    axes[idx].set_xlabel('Tenure (meses)', fontweight='bold')
    axes[idx].set_ylabel('Monthly Charges ($)', fontweight='bold')
    axes[idx].set_title(f'Tenure vs Monthly Charges por {demo_var}', fontweight='bold')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Relação: Idade (SeniorCitizen) vs Custos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Monthly Charges por SeniorCitizen e Churn
senior_monthly = df.groupby(['SeniorCitizen', 'Churn'])['MonthlyCharges'].mean().unstack()
senior_monthly.plot(kind='bar', ax=axes[0], color=['green', 'red'], alpha=0.7)
axes[0].set_title('Média de Monthly Charges por SeniorCitizen e Churn', fontweight='bold')
axes[0].set_ylabel('Média de Monthly Charges ($)')
axes[0].set_xlabel('SeniorCitizen')
axes[0].legend(title='Churn')
axes[0].set_xticklabels(['Não', 'Sim'], rotation=0)

# Total Charges por SeniorCitizen e Churn
senior_total = df.groupby(['SeniorCitizen', 'Churn'])['TotalCharges'].mean().unstack()
senior_total.plot(kind='bar', ax=axes[1], color=['green', 'red'], alpha=0.7)
axes[1].set_title('Média de Total Charges por SeniorCitizen e Churn', fontweight='bold')
axes[1].set_ylabel('Média de Total Charges ($)')
axes[1].set_xlabel('SeniorCitizen')
axes[1].legend(title='Churn')
axes[1].set_xticklabels(['Não', 'Sim'], rotation=0)

plt.tight_layout()
plt.show()

# Correlação entre Tenure e Churn
print(f"\n📊 Correlação entre Tenure e Churn:")
churn_numeric = (df['Churn'] == 'Yes').astype(int)
corr_tenure_churn = df['tenure'].corr(churn_numeric)
print(f"  Correlação: {corr_tenure_churn:.4f}")

# Relação: Partner/Dependents vs Custos
fig, ax = plt.subplots(figsize=(12, 5))

family_status = []
for _, row in df.iterrows():
    if row['Partner'] == 'Yes' and row['Dependents'] == 'Yes':
        status = 'Com Parceiro + Dependentes'
    elif row['Partner'] == 'Yes':
        status = 'Com Parceiro (Sem Dependentes)'
    elif row['Dependents'] == 'Yes':
        status = 'Sem Parceiro + Dependentes'
    else:
        status = 'Sozinho'
    family_status.append(status)

df['FamilyStatus'] = family_status

family_summary = df.groupby('FamilyStatus')['MonthlyCharges'].mean().sort_values(ascending=False)
family_summary.plot(kind='barh', ax=ax, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_title('Média de Monthly Charges por Status Familiar', fontweight='bold', fontsize=12)
ax.set_xlabel('Média de Monthly Charges ($)')

for i, v in enumerate(family_summary.values):
    ax.text(v + 1, i, f'${v:.2f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Matriz de Correlação

In [ ]:
print("=" * 80)
print("MATRIZ DE CORRELAÇÃO - VARIÁVEIS NUMÉRICAS")
print("=" * 80)

# Calcular matriz de correlação (incluindo Churn como numérico)
df_corr = df[numerical_cols + ['Churn']].copy()
df_corr['Churn_numeric'] = (df_corr['Churn'] == 'Yes').astype(int)
df_corr = df_corr.drop('Churn', axis=1)

corr_matrix = df_corr.corr()

print("\n📊 Matriz de Correlação de Pearson:")
display(corr_matrix.round(3))

# Identificar correlações altas
print("\n🔍 Correlações Altas (|r| > 0.5):")
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.5:
            high_corr.append({
                'Variável 1': corr_matrix.columns[i],
                'Variável 2': corr_matrix.columns[j],
                'Correlação': corr_matrix.iloc[i, j]
            })

if high_corr:
    high_corr_df = pd.DataFrame(high_corr).sort_values('Correlação', key=abs, ascending=False)
    display(high_corr_df)
else:
    print("  Nenhuma correlação encontrada com |r| > 0.5")

# Correlação com a variável alvo (Churn)
print("\n🎯 Correlação com Churn:")
churn_corr = corr_matrix['Churn_numeric'].drop('Churn_numeric').sort_values(key=abs, ascending=False)
churn_corr_df = pd.DataFrame({'Variável': churn_corr.index, 'Correlação com Churn': churn_corr.values})
display(churn_corr_df.round(4))

# Heatmap da matriz de correlação
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, ax=ax, cbar_kws={'label': 'Correlação'}, 
            vmin=-1, vmax=1, linewidths=0.5, linecolor='gray')

ax.set_title('Matriz de Correlação - Variáveis Numéricas', fontweight='bold', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

# Correlação com Churn - Visualização
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['red' if x < 0 else 'green' for x in churn_corr.values]
churn_corr.plot(kind='barh', ax=ax, color=colors, alpha=0.7, edgecolor='black')

ax.set_title('Correlação de Variáveis Numéricas com Churn', fontweight='bold', fontsize=12)
ax.set_xlabel('Correlação de Pearson')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)

for i, v in enumerate(churn_corr.values):
    offset = 0.02 if v > 0 else -0.02
    ax.text(v + offset, i, f'{v:.3f}', va='center', 
            ha='left' if v > 0 else 'right', fontweight='bold')

plt.tight_layout()
plt.show()

## 11. Resumo Executivo e Conclusões

In [ ]:
print("=" * 80)
print("RESUMO EXECUTIVO DA ANÁLISE EXPLORATÓRIA")
print("=" * 80)

print(f"""
📊 DATASET
  • Total de registros: {df.shape[0]:,}
  • Total de features: {df.shape[1]}
  • Variáveis numéricas: {len(numerical_cols)}
  • Variáveis categóricas: {len(categorical_cols)}

🔍 QUALIDADE DOS DADOS
  • Valores nulos encontrados: {df.isnull().sum().sum()}
  • Percentual de dados completos: {(1 - df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100:.2f}%

📈 VARIÁVEL ALVO (CHURN)
  • Clientes não churn: {churn_counts.get('No', 0):,} ({churn_pct.get('No', 0):.2f}%)
  • Clientes churn: {churn_counts.get('Yes', 0):,} ({churn_pct.get('Yes', 0):.2f}%)
  • Dataset {'BALANCEADO' if abs(churn_pct.get('No', 0) - churn_pct.get('Yes', 0)) < 20 else 'DESBALANCEADO'}

🔢 VARIÁVEIS NUMÉRICAS
  • Discretas: {len(discrete_numerical)} ({discrete_numerical})
  • Contínuas: {len(continuous_numerical)} ({continuous_numerical})
  
⚠️  OUTLIERS DETECTADOS
  • Variáveis com outliers: {sum([1 for x in outlier_df['Qtd. Outliers'] if x > 0])} / {len(numerical_cols)}
  • Percentual total de outliers: {(outlier_df['Qtd. Outliers'].sum() / len(df)) * 100:.2f}%

🎯 SEPARABILIDADE DE CLASSES
  • Variáveis numéricas significativas: {sum(sep_numeric_df['P-Value'] < 0.05)} / {len(numerical_cols)} (p < 0.05)
  • Variáveis categóricas significativas: {sum(sep_categorical_df['P-Value'] < 0.05)} / {len(categorical_cols)} (p < 0.05)

🔗 CORRELAÇÕES
  • Pares altamente correlacionados (|r| > 0.5): {len(high_corr)}
  • Variável mais correlacionada com Churn: {churn_corr.idxmax()} (r = {churn_corr.max():.4f})
  • Variável menos correlacionada com Churn: {churn_corr.idxmin()} (r = {churn_corr.min():.4f})

💡 PRÓXIMOS PASSOS RECOMENDADOS
  1. Realizar feature engineering para melhorar separabilidade
  2. Tratar outliers se necessário para alguns modelos
  3. Codificar variáveis categóricas para modelagem
  4. Considerar balanceamento de classes se aplicável
  5. Testar diferentes algoritmos de classificação
""")

print("=" * 80)
print("✅ ANÁLISE EXPLORATÓRIA CONCLUÍDA COM SUCESSO!")
print("=" * 80)